# Predictive Wildfire & Deforestation Mapping

In [14]:
import glob
import rasterio
import numpy as np
import pandas as pd

# 1. Define the years and your main project folder
years = ['2024', '2025', '2026']
base_folder = r"C:\Users\hp\Downloads\Project"

# This list will hold the finished tables for each year before we glue them together
all_yearly_tables = []

for year in years:
    print(f"--- Starting processing for {year} ---")
    
    # --- Step A: Find all Sentinel-2 images for this specific year ---
    search_pattern = rf"{base_folder}\Sentinel2_{year}*.tif"
    image_list = glob.glob(search_pattern)
    
    if len(image_list) == 0:
        print(f"No Sentinel-2 images found for {year}. Skipping.\n")
        continue # Skips to the next year
        
    print(f"Found {len(image_list)} images. Stacking and squashing...")
    
    red_stack, nir_stack, swir_stack = [], [], []
    
    for file in image_list:
        with rasterio.open(file) as src:
            red_stack.append(src.read(3))  # Band 4 (Red)
            nir_stack.append(src.read(7))  # Band 8 (NIR)
            swir_stack.append(src.read(9)) # Band 11 (SWIR)
            
    # --- Step B: The "Squash" (Calculate median across time) ---
    red_flat = np.median(np.array(red_stack), axis=0).flatten()
    nir_flat = np.median(np.array(nir_stack), axis=0).flatten()
    swir_flat = np.median(np.array(swir_stack), axis=0).flatten()
    
    df_yearly = pd.DataFrame({
        'Red_B4': red_flat,
        'NIR_B8': nir_flat,
        'SWIR_B11': swir_flat
    }).astype(float)
    
    # --- Step C: Engineer Features (NDVI & NDWI) ---
    df_yearly['NDVI'] = (df_yearly['NIR_B8'] - df_yearly['Red_B4']) / (df_yearly['NIR_B8'] + df_yearly['Red_B4'] + 1e-8)
    df_yearly['NDWI'] = (df_yearly['NIR_B8'] - df_yearly['SWIR_B11']) / (df_yearly['NIR_B8'] + df_yearly['SWIR_B11'] + 1e-8)
    
    # --- Step D: Attach the FIRMS Fire Label ---
    firms_path = rf"{base_folder}\FIRMS_Fire_Labels_{year}.tif"
    try:
        with rasterio.open(firms_path) as src:
            df_yearly['Target_Fire'] = src.read(1).flatten()
    except FileNotFoundError:
        print(f"Warning: No FIRMS label found for {year}. Skipping this year.\n")
        continue
        
    # --- Step E: Clean the data (Drop empty background pixels) ---
    df_yearly_clean = df_yearly[df_yearly['NIR_B8'] > 0].copy()
    
    print(f"Finished {year}: {len(df_yearly_clean)} valid pixels extracted.\n")
    
    # Add this year's finished table to our holding list
    all_yearly_tables.append(df_yearly_clean)

# --- Step F: Combine all years into one massive dataset ---
if len(all_yearly_tables) > 0:
    final_master_df = pd.concat(all_yearly_tables, axis=0)
    
    print("=========================================")
    print("SUCCESS! Final Master Dataset Created.")
    print(f"Total Rows (Pixels): {len(final_master_df)}")
    display(final_master_df.head())
else:
    print("No data was processed. Please check your file paths and downloads.")

--- Starting processing for 2024 ---
Found 16 images. Stacking and squashing...
Finished 2024: 3836004 valid pixels extracted.

--- Starting processing for 2025 ---
Found 18 images. Stacking and squashing...
Finished 2025: 3836004 valid pixels extracted.

--- Starting processing for 2026 ---
Found 19 images. Stacking and squashing...
Finished 2026: 3836004 valid pixels extracted.

SUCCESS! Final Master Dataset Created.
Total Rows (Pixels): 11508012


,Red_B4,NIR_B8,SWIR_B11,NDVI,NDWI,Target_Fire
3998322,569.0,2074.5,1313.0,0.569510,0.224797,0
3998323,581.0,2010.5,1321.5,0.551611,0.206783,0
3998324,580.0,1995.5,1321.5,0.549602,0.203196,0
3998325,577.0,1970.0,1413.0,0.546918,0.164647,0
3998326,577.0,1916.0,1413.0,0.537104,0.151096,0


In [15]:
final_master_df['Target_Fire'].value_counts()

Target_Fire
0    11387246
1      120766
Name: count, dtype: int64

We can clearly see that data is biased towards 'no-fire'. So, we will do undersampling in order to select same number of samples for both the values so that model can distinguish between the 2 classes

In [16]:
!pip install imbalanced-learn

In [17]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd
import numpy as np

In [27]:
X = final_master_df[['Red_B4', 'NIR_B8', 'SWIR_B11', 'NDVI', 'NDWI']]
y = final_master_df['Target_Fire']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

In [31]:
undersample = RandomUnderSampler(random_state = 42)
X_train_balanced, y_train_balanced = undersample.fit_resample(X_train, y_train)

lr = LogisticRegression(max_iter=1000, random_state=42);
lr.fit(X_train_balanced, y_train_balanced )
y_preds_lr = lr.predict(X_test)
y_preds_lr

array([1, 1, 1, ..., 0, 0, 0], shape=(2301603,), dtype=uint8)

In [32]:
SVM = LinearSVC(penalty = 'l2', C = 10.0)
SVM.fit(X_train_balanced, y_train_balanced )
y_pred_SVM = SVM.predict(X_test)

In [34]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, fbeta_score

f5_scorer = make_scorer(fbeta_score, beta = 5, pos_label = 1)
param_grid = {
    'n_estimators': [10, 50, 100, 200]
}

GS_RFC = GridSearchCV(RandomForestClassifier(random_state=42), 
                      param_grid = param_grid,
                      scoring = f5_scorer,
                      cv = 3, verbose = 2)

GS_RFC.fit(X_train_balanced, y_train_balanced )
GS_RFC.best_estimator_

Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END ....................................n_estimators=10; total time=   6.8s
[CV] END ....................................n_estimators=10; total time=   7.3s
[CV] END ....................................n_estimators=10; total time=   7.1s
[CV] END ....................................n_estimators=50; total time=  35.6s
[CV] END ....................................n_estimators=50; total time=  36.0s
[CV] END ....................................n_estimators=50; total time=  35.8s
[CV] END ...................................n_estimators=100; total time= 1.2min
[CV] END ...................................n_estimators=100; total time= 1.2min
[CV] END ...................................n_estimators=100; total time= 1.2min
[CV] END ...................................n_estimators=200; total time= 2.4min
[CV] END ...................................n_estimators=200; total time= 2.4min
[CV] END ...................................n_est

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [36]:
RFC = GS_RFC.best_estimator_
RFC.fit(X_train_balanced, y_train_balanced)
y_pred_RFC = RFC.predict(X_test)

In [37]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 1]
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    scoring=f5_scorer,
    cv=3, 
    verbose=2, 
)

grid_search.fit(X_train_balanced, y_train_balanced)
grid_search.best_estimator_

Fitting 3 folds for each of 9 candidates, totalling 27 fits
[CV] END ................learning_rate=0.01, n_estimators=50; total time=  18.2s
[CV] END ................learning_rate=0.01, n_estimators=50; total time=  18.1s
[CV] END ................learning_rate=0.01, n_estimators=50; total time=  18.3s
[CV] END ...............learning_rate=0.01, n_estimators=100; total time=  37.1s
[CV] END ...............learning_rate=0.01, n_estimators=100; total time=  36.7s
[CV] END ...............learning_rate=0.01, n_estimators=100; total time=  37.2s
[CV] END ...............learning_rate=0.01, n_estimators=200; total time= 1.2min
[CV] END ...............learning_rate=0.01, n_estimators=200; total time= 1.2min
[CV] END ...............learning_rate=0.01, n_estimators=200; total time= 1.2min
[CV] END .................learning_rate=0.1, n_estimators=50; total time=  19.1s
[CV] END .................learning_rate=0.1, n_estimators=50; total time=  33.8s
[CV] END .................learning_rate=0.1, n_es

,loss,'log_loss'
,learning_rate,1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [38]:
GBC = grid_search.best_estimator_
y_pred_GBC = GBC.predict(X_test)

In [41]:
leaderboard_data = []

def grade_model(model, true, predicted):
    precision, recall, f5, support = precision_recall_fscore_support(
        true, predicted, beta=5, pos_label=1, average='binary', zero_division=0
    )

    leaderboard_data.append({
        "Algorithm": model,
        "Recall": round(recall, 4),
        "Precision": round(precision, 4),
        "F5-Score": round(f5, 4)
    })

grade_model("Logistic Regression", y_test, y_preds_lr)
grade_model("Support Vector Machine", y_test, y_pred_SVM)
grade_model("Random Forest", y_test, y_pred_RFC)
grade_model("Gradient Boosting", y_test, y_pred_GBC)

leaderboard_df = pd.DataFrame(leaderboard_data).sort_values(by="F5-Score", ascending=False)

In [42]:
leaderboard_df

,Algorithm,Recall,Precision,F5-Score
3,Gradient Boosting,0.7183,0.0244,0.3428
2,Random Forest,0.7069,0.0232,0.3313
0,Logistic Regression,0.5237,0.0260,0.3017
1,Support Vector Machine,0.5529,0.0237,0.2976
